# 05 - Đánh giá baseline RT-IoT2022

## Mục tiêu
So sánh các mô hình baseline trên validation, chọn mô hình có Macro-F1 cao nhất, sau đó đánh giá một lần trên test. Notebook không huấn luyện lại mô hình.

## 1. Nạp dữ liệu và các mô hình đã huấn luyện
Tất cả đường dẫn được lấy từ cấu trúc dự án chuẩn để tránh phụ thuộc vào thư mục chạy notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'raw' / 'RT_IOT2022.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy thư mục dự án chứa data/raw/RT_IOT2022.csv.')
HELPER_DIR = ROOT / 'src' / 'baseline'
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score

from baseline_utils import baseline_paths, load_processed_split
paths = baseline_paths(ROOT)
MODEL_NAMES = ['Decision_Tree', 'KNN', 'MLP', 'Random_Forest', 'SVM']
X_val, y_val = load_processed_split('val', paths['splits'])
X_test, y_test = load_processed_split('test', paths['splits'])
models = {}
for name in MODEL_NAMES:
    model_path = paths['models'] / f'{name}.joblib'
    if model_path.is_file():
        models[name] = joblib.load(model_path)
if not models:
    raise FileNotFoundError('Chưa có model baseline. Hãy chạy các notebook trong 04_Training trước.')
print(f'Models: {list(models)} | Validation: {X_val.shape} | Test: {X_test.shape}')

## 2. So sánh trên validation
Validation được dùng để chọn mô hình. Test chưa được sử dụng ở bước này nhằm giữ nguyên vai trò đánh giá cuối cùng.

In [ ]:
validation_rows = []
for name, model in models.items():
    prediction = model.predict(X_val)
    validation_rows.append({
        'model': name,
        'macro_f1': f1_score(y_val, prediction, average='macro', zero_division=0),
        'macro_recall': recall_score(y_val, prediction, average='macro', zero_division=0),
        'accuracy': accuracy_score(y_val, prediction),
    })
comparison = pd.DataFrame(validation_rows).sort_values('macro_f1', ascending=False)
comparison.to_csv(paths['metrics'] / 'validation_model_comparison.csv', index=False)
display(comparison)

fig, ax = plt.subplots(figsize=(10, 5))
comparison.plot(x='model', y=['macro_f1', 'macro_recall', 'accuracy'], kind='bar', ax=ax)
ax.set_title('So sánh mô hình trên validation')
ax.set_ylabel('Điểm số')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(paths['figures'] / 'evaluation' / 'validation_model_comparison.png', dpi=140)
plt.show()
selected_name = comparison.iloc[0]['model']
print('Mô hình được chọn:', selected_name)

## 3. Đánh giá cuối cùng trên test
Chỉ mô hình được chọn từ validation được đánh giá trên test. Kết quả được lưu cùng classification report để phục vụ báo cáo.

In [ ]:
selected_model = models[selected_name]
test_prediction = selected_model.predict(X_test)
test_result = pd.DataFrame([{
    'model': selected_name,
    'macro_f1': f1_score(y_test, test_prediction, average='macro', zero_division=0),
    'macro_recall': recall_score(y_test, test_prediction, average='macro', zero_division=0),
    'accuracy': accuracy_score(y_test, test_prediction),
}])
test_result.to_csv(paths['metrics'] / f'{selected_name}_test_metrics.csv', index=False)
pd.DataFrame(classification_report(y_test, test_prediction, output_dict=True, zero_division=0)).T.to_csv(
    paths['metrics'] / f'{selected_name}_test_classification_report.csv'
)
display(test_result)
print('Đã lưu kết quả vào:', paths['metrics'])